# 🎬 Montage automatique Locapay

Chargez une **vidéo brute face caméra** : le notebook la **transcrit**, **coupe les blancs et les hésitations**, ajoute les **sous-titres animés mot par mot** (italique fin + mot clé en gras capitales), les **recadrages**, les **écrans de couleur Locapay**, le **logo**, des **effets sonores légers** et, si vous voulez, une **musique de fond** qui baisse quand on parle. Vous récupérez un MP4 vertical 1080×1920 prêt à publier.

**Mode d'emploi**
1. Vérifiez que le GPU est activé : menu **Exécution › Modifier le type d'exécution › T4 GPU** (gratuit).
2. Menu **Exécution › Tout exécuter**.
3. À l'étape 3, cliquez sur **Choisir des fichiers** et sélectionnez votre vidéo (et votre musique si vous l'avez cochée).
4. Le téléchargement de la vidéo montée démarre tout seul à la fin.

💡 Si un mot est mal transcrit (ex. « Locapé »), corrigez-le à l'étape 5 puis relancez seulement les étapes 5 et 6 (bouton ▶ à gauche de chaque étape).

In [ ]:
#@title 1️⃣ Installation (1 à 2 minutes)
import os, subprocess, base64, gzip, sys
os.makedirs("/content/montage/polices", exist_ok=True)
subprocess.run("pip -q install -U openai-whisper", shell=True, check=True)
P = "/content/montage/polices"
for nom, url in [("serif.ttf", "https://raw.githubusercontent.com/google/fonts/main/ofl/playfairdisplay/PlayfairDisplay-Italic%5Bwght%5D.ttf"),
                 ("gras.ttf", "https://raw.githubusercontent.com/google/fonts/main/ofl/montserrat/Montserrat%5Bwght%5D.ttf")]:
    subprocess.run(["wget", "-q", "-O", f"{P}/{nom}", url])
open("/content/montage/montage_auto.py", "wb").write(gzip.decompress(base64.b64decode("H4sIAAAAAAAC/7193ZIbR5befT9FbckTXRDRINBNtkhwoDUlURS9lMRgc0YbhmBMASh0F4k/VRW6m2x3xLyBr9dXEw6HdzQbsd4Lh+2N8ZX6TfYF/Ar+vnMyq7J+utkcac2QGkBWVubJkydPnjx/6fv+ztfrVRYeR164zdbLMIt/2EbeT//gpdnbReQ9X0/DTfjW++nPO3u3+bfz23h29ePamyTbLPLm4TTypuHy6sck9Ly9Tz0vS8JVOk3iTRavV17w3UmcbqKkpQ/x5nS93UTeLEq9ySJcTdMdlqfrbbqXxVmC4nAVo7nUW64zbxMm8hnEWbgQuOfxyrsjZdPF1Y9etPKOkzAFCBtWidKWtJdE03CWYMxobpFFyYoN3vHwxnGUeO/W66U3BVLi1VbhAlgEG2ChfLuItgm/LsMEXUqDi/XxWqtG83mUpYB4tSa02mSqz5bbVIDEu/P1auYFa8HCKlosorY3207fxKtjhfDrF/e80yjJ4mm48HrdB93z3sP9rrdJrv6UeVd/8DbbySIGrMGdCkY72XnmRZnXSZOs1dnxMb/xcrNOMu81YGoD6Oyk7a3TNpDQ9tKTbRYv8LmdbJL1NEpRfhaeRjvzBDiYhVk4XYRpioGYRvKiNlAdLWY7tvXVdrl56wHVq42+/OLZc/vSsyVQ3daPL5LwzHz9Mibu7Q/ge2fnu7b3Vdv78sWRN5BRtz0Ou+0ddHeOXqLs3oNut7uzs/ORN/gZ/7wEs7Lg9O/823xAO/LXexnpk/6Oh39mtse9PpZDAgD8j/af7H9+v+d7zr+PQKvRFiiIs3y9BJZm7pAmW6XW9ovWuo8/3398r9IaJ942JC9u1mnM2R1zIYx1IfS9+WIdZmil2zk81Be7+HWCZQy04dskTOXt1WQcCSxjA0Dfi1d8cV8eK8WODcXmoCnl+g5YedF66/nhdrpd+dLA6XqxXUZjQ90uXL0HUiHdbjZJvIyS8UmEgYQcCjqarNcLVHuVYElIB9EWpHmyXbYB+kmn09Gxh9sUjYfnbsP37nvlfx+Be+3NosXVH9oeuJbwDi9KM+J8A0YQpDoFWLPH0Tg8DVdZCdCu+3RTQW/vvjydbZMoGm/QdBmc/c69ayDZrgpWI9DEqxSs8OpHaY98pmilZ7uJwKjAFPBKCUXyjHxmHK0wkfGq/lQBnPNR0apOwTyenoBfjNlAPsW+Ia5FPI3GaZTE8+ZHZKFpVHn2kQduD4a0nscLcjTZLuRJFsZgaOMsOs8iF5JuZcaW20UWb9BDmBmOugjNy7IBOFxfZ2cNMgXLBzzcJCwV36tSwmq9nCRkz+fCpfGWF86JAHL5aJVGy4mBFM/G00VUI8Vqi5hHd0t5+vLxkff54xfPXj1+/uRIdiEFSVd5CVlYVEtsPH69UaknA8WK5V4hkPY984Z3FyN4vU2nWLIpf2A30h/SC7dBWUh5T5v1pqGXcAPwhHlIV6YP1EWT3IS2+JS1rMDLLI5dEjStT8FXZn6B4zADsS3wsjzgJxg46CQDitByso5ZiA09ZU1D8cKGlECrtKvkhLnfokPlEF7w2V6yXixaAHh44c+iid/2fLzte0EWLTdCIck0akmp0DcrZG83kQ/i4K7yH0/jWbS+HEkXpcb73iJOST6yjQWzaB6CHMcQWLJ18nbAh8oxZljA3BGa+FvXQTRoN9/fr34EJwEJYqOeRhnkoDQCekCMQPdyDZDDLTaxJyCfb7/55vHTJ9zuLqQ33yAWA/CjHwYWqQOsnoO+oFLAl99+W1/Rqam/cFh54UH/OFwuw0G38/BBe7perJNJSD4ZDZIUhehgkg725EuyZMF+f7KUgv28q3yS6/317pf76+0X/R3aBoQwGmCtvNutwbdnAcTnPcLlglWQWUPTD9ymu51P+NYl0I8p906i8/3keBKctHS/P+EG2gGRxJug1VnoF/8jX0khidDOysu2m0UUgPkEJ8O4H2OH3x9hxz1sYUElXgy25AWQWfbb3r1W6+cLKyX5TqAeCyecRZttjH07Ol5GqywN7BczFFnpA2+otE/IUIGw2XpazT48M486x1EW+GfrZJZiMQ1HLQ8Ph6OisnB4NAzcBGdDqemPWhZnpWrx3MvKLwpgJHb/DDMFOcVPfbOg2FiahUmG1lAeueXRin1c1ppCB2cKMPahSTiJF3H21m95McRQ8Otv1quo3r/AMPQ3/giAgG2vZkHekdsMwTho1cEHXjvgqoApWJbogk8MXeErBb3S1AVmYiCSLxfnXvB1OPUeb0BK3hE6BMWivzloNkr2zvRUpHWerTJhtC8+R4U1+g3josLn60U4aXV8sykbgVs/FvGko+I9ZJYMk7zcMUizvzvpW/S3DFqgUM//IkzOyFxDHE3yGssQjHUVmSphsjy8pzXKfXTAlmdjwDQNOLqxAdBvFfg3WOJj3wJyfSOKiZva0Rq3aOqGNhSd2kYSxtiPX25x7ltGT5JknQT+Y+6MZjYpTJTPWpDmsFFSKAgwKDsptVmEuFyeNswX+cLX337xBPLD+OvnfytLAiBN1mRgbGy6Xi63K9DhXfPS3oLC6d7pwZ7W49qBnLC4/gV5vEd0t6tU7C+jWbxdXv+uPteXIfibrm8BHN+4zIf25eOjV09elkd3wzj0S7sEnn6rAJF/t4zcTkwSBbLl46C7hhAeDXzbC3aS4y1+zykknOJwNdkuQtQf+H6xMF+V5tcoJzwciKfeyTpZ87B4HJVVD4Y4pifrOI3BQSE4iSyg66ZYmqbe4BreYMnYVhvoQilI1ixtZ3Hlj3C4wmY0r0BPmmx7F4qHy9a//P6/YeBziEAnA0pdBWcToIpmOxaySY7LDZQG4zWOb/NxEm3WA4dyhf1qHxbnLYNsoGqgWG83MuHKP+4kY649LCoIdwJkG2sMwmu4GIMzLzfZwJk4bkzk8LdqnOKXHqHxH853p7GcpHFEGXwZLlIHG1nytq8CXTy5+mMihxIonJZrdiny2+zqj2DaAdkMJDtMFlgEGfUD7ylwJWdLnkGXRq2G5b/YffYYZzcIklF5QxFNSTPu7YR/TZx+tV7MnAnnP6e8I3hv14vGnDjMrqBJ/pb3Z+3heNomDJ0pzv/U4CzPS7WOp3iCM9k0q2zvAaY+zCAGLEFnPs5ROJpNQfaRr32J6LA87yx5nOg4z1tOQ9H5NNpk3hP54HGm1MWGipkK026WfhJID+Y7du6G1WS2C0fqIe7L24zFiFn4gs/3rbIys7/VgqME5HYRlNll85KagWYhCvvU0qJdsOANNKxjnnYGPiB74BcdABNQ0I25rhtWc3Vx/uutvNuvuhvnttQNToIqnfbtV3w76/ALtxKRIFkg3yhIrrQCPvHLFfBY6vy+vHY0uXwcpNJPqjJxa3SpcrURnVNLd0pEOEdOT9wCl21jNuVUvZ2FvsjJrNzhzw5GDh4SQ65bUOyKgCdU3GxzSUfeHWiZiGJKIwJgvt3pFlnaN4t91ZGElKr9f/nPf/i///s/ed6LULTcT1/8Boduy+p2ZV/e9cjx4pTizgZUDR63ElUzWCMO61BN/ziN01bH8yuI9B9Ps/g0ekc+qQ0/OUfdrayfT8m14jl12eiJ1OzNdiP3+at7fKvjkLcZ78AKDDvu8vz8hOMl4bjc2m7lsGvYBQqThnDvC+DzkovVaAjRruXGOBbMxlI9X49mGeLDyP4/a5E1k9wtVp433/QOB0FOCqSj1jXNvW8VtqlwnqzTyF2UH8Jsd3a+enIEZdirZ99+I6oMH5pckttJtOUHfulPFeROlvKxXEqh/oXCVz7Mi9EJJLujV9++4EGT5zvfX8hOTF3YgoQEhZUotFSpMiMdYqNdQob3ZmvofoGoVQwdW+K9BlltofzxaGbxQFIrDB8SIP7Ei1RK050lKlFNQ2LGYoy9xRb/c+/gn9R7S70fbVnoR1U6MJZ4V38fogtv6v2A2tT8/LDFf3gbIACeq38mTCHVQCE+t+c71AGnfJYCn57oOJdLNvSnjHrAmNtveErdV8gFw5oYJ3ZBT1Zcllz9kX8j/oEKbwP9l0iiIqGmNDeQoqk43ZnxFzbbd5xdKv6guMD/6B1YgpoV1T12zY5p4MGBGfR1Kn9XayII5TH3GeAG6sCEesoUJZM4YlPbbCcjDvkt0r8p9ClgQmsgY4VuTtcxtODygYnDN3zE05iDpaCB8gT7L2wmcyHt6a5gZxcz9cPW/OXk7LzeBWJW8nS2i6nnHxDDLrrA2JcxpkYbpOAF5T1xJDp8T+nW41mdz6KViDjQFr0GNLPodeid4tRHOt6BphXDWF79CaNdgn2gd7JWGT5eJaZlZjfRNuOfc6BV/wCqmP8nEai0k0KhDUGptfPl42efPRfN3gcTMOnFUAzJbmbntWG6C8LbQXsrEvvr95N7nZKnH0QgO5ZA3nLJgLiAM66fzKVQodicgtyZyqmyhDA9yFH7HWRmgzJMKIk6MFAGiT/8D9+fXf3h6r9c/derH6/+ePWnq3+4+ser/371P67+59U/X/356v9c/f3uv/z+7/ZGZCH4PwMDP4sSNN7BsWUBWzQ2ut//HZ/u+rZDqHLEdAzZdA2d7VS0wzzVpRSy8pLijOgUYivbpVgPC9PA2OseGe0vxrhNBs73dNeeBDPu7GSTl7lqbiMrINahCkIw2Effr0Z+CQhyNd/dvSEN+AOfL0oLZeE5hB0N3cgT0yoqQ1NZ09KFVn1X15YJsEOZlbxWi5qzSUnlF89F4ya1a2oWUYzxN5hEWSW5JOh8XLzzBjWkO+jozqwob7p4w+qVPtROtZqyZYyTeh/IBMRfu/NXf/2oj519dOffYNjV9uxmP4txsoHN8WygY30zgkY3kCaH3ZGoy6R5FcT8VtEChtOsCSRzVOrieRlyaJwEYg6j/DCh6RyWeGmVxpBw0aCEoPItIp+Q98gGRKFrtBC0ZEHekqaVNwxtwyNIqaqEkN1rTeljmxR6CLCD04icZaBN2wXoTqOtU5tJM3UWJwZ6EUL5HuksIr2X0U/s2yaHe72RQ2ZusXdnUGrWmCDjmeixIW0GZy0q2wt53L6tUGVrnDPb1CornaXbZSCvtwzaHcKDD8JGmngHErbNtLU3B7wZmgnoU7LHFlrex3zrrvbURAaq1z4zem1VLmdUIxuNtimhi0p4HsCEcR+goVWYSPZbpt4G9WC2uHRUEsTM7BoCE4wBMwr05n2r4KxEqBtL4Za6TavkOTFs/TfxwceGccpORntZpP4xDneUqjAS4/Rz9aM4FS3Xsp1C9AYyYMMjU5Y9Uah79dM/wX6C7e811CtodT2dbtEcTED2eIEVpJq2SLdF1tqcwNQDbcOjencijLM3bK34CSnl9ZriXKV/HDgoL6Q8cPxO5IjfEQC+xV0sFhGQELa4j1HWUd+E36zyRmRY7C73diiGFXDjhX2d4pz6I1CWsgqikopQSF05UqvMIZV2pzFYVCS1pvJ8yueVLWIoczo1lhFgI4p81d8r+v1iHcjrKavMAtN223sTwQAaLicziLZ9b4/nqunQj8E83a2njVkaeM7Dtv3xusK2yRqCrvdr1Oaf196veXYU2mpVthzjdlXoqdJNuFKlaQpj22usnN5ox7XF5KWoVXBbM3gRA0UaAHRsCvSOs4s/Mr/AdWBd4k+zjopS7hWtMp5qaC5txDfg+z1jFMRzSjPpK5OTvOyB50oE5zJLWTEskqgOq5ULUKJGKHBDrBHL0nhtOlbvAWlNJ6I3slm3vW5ejIMxIH3jTmC/Kky8wWysOM95FdkfVIaQGWONc4PiYnzY9Y6jYEXhYqAoqUsjLjW86UtPo1olcv87/1rE4P57w25WpWKy0X4TRLmgIKCPWo/07V6JljE0cvZmm6Ls1hWzgTMD1qTl+Z3XUMgY8UmUoDr8lBQjikjD7IfcFPwyl/kF7NWncMWchbnHCVW44mp39OQ3z56PP//2my+fPf7m8yfquGZ3sddRRrFbxlge0M2LoiTC6BIQasuEylHB0UyMrMQP4eJtmm9uabSNF4MKdMU294X1SaVzUkn1C5lMt56Uh7c0PhZZaKonLB584eqlmq18I1RcAC6qbxOIbgFW3jymk0OuosJpmOygTxfaGKtQvIOM+cocphbiQIrpcza9hNZLyoRXP8ruF4jdgsdZBVL8zxLo/7BP62HXHJX7PCRC1QQnQ9nBj8NkRm1ciB/JKqYNpOhmTslxppKnF+DLhOffvI9wAuTDcwg6Qjl5Ywkm1AnaHc7gC5u4mMLIXByhDFuKEGIEZ9IIDhtRlcOQy8J9sSz7J6JKlXWFh1yvoypL4mLAOhc0RaeyyKFYBPmRjZeEVghLVmLVqrUjQ76mE54oImk177w7Uo6C1nv7DacpM37LDPAKGIHFRbEfbBNbI87l8tLAG9rZaT5hvcGuPDsvY9W87cB3JhKFbqejwo0Fb44cTfF6ItXO1Hkil8LxsuOIUXa/cAYFQrdCOVtqQY7mDmF+UAiVDoQ3lUxVuiBS83JPD8xy5pUXRHz9NWbz4P6NbRa7GJZpkkVpGelEMUF0gBdOIisU7Qub4KxbcD4daJ+VQ7e2bWfGd5a6rGfRCLor+q9bfqv5KLHCQYBbXA9bFp3fcLBIiy1LHOGkqNi3fEwYXd9m582qYF/YJQ3odpc4013CmUseQThk1OIHfpkhocB8u6ZxZRx+X/Bnq9IkUy66NKT6kfKs1GFa9HN0OQyZE9EX0jcDXEr0XsT9Qc55oUilMGS8ItVtPZ7PYby1ZpejCIcUMNivwwyKmMRdGWF5XQD7zmxCOzjIN6ZwaHBXlmxJbK/DFshj/z0CFXuceKIdzUQA6on4cm9Uk58mQ5lt8hX0ylkmO9m/X+cm8HIN35RKX08cmCcNMIsQFVPtUUFLoGaF11AavYZDkbjLVey9gGqbjZWlG+FSxm6+TlTOez0Z9s0jkeRehzVuTABk9RzKG7YhFB2IzdjtqD5qIMVS5MgulrmvtSM51lEfm2/O3sVkuLvaHV36VvZ1m7+N74Cyj7lf2vv6Zm9r6qjVBLNZHTymiJnlhtksHfI//5ZOojAO//bx82dfiCRDxuV/89M/UZb9QrZgfvsSlIKPV6Ivor0KsgcVWi2Wfm7FDP64+n0hm/D3UwUN344Kl2vfiksLCDZROi7Eusr+YeW04WYIlgUupEqOjSFj6hudMqFnLdvk9FlhKL5MFR4LHxqRJQs7NvNwoY31ul3qY4a7rLUL90DvV1VHp9yu6WOp+d5P/8uyvY1DQyo2uiVWGSIg2nmTH+Blo5GzASkiLKbGtCZAPC3hhce281bhU6oaaSue0tENnRH3Xf6B3UQ/jAqbBVGDrtqZDZVic8lKp6tBTSN6f+hOxCt+savGXHW97xchGm0r/6k1qaAHatWMMlK1Q8anO4jOOxSMEb4kUvEcMjLsYlyLGRXjkZottqtc+sWKmTMUySow1YmG8iY9eMUcYqoWgmNNrtmcmKFaJV4FAcZEX7APrQx1HlVVgfyiNnEoUgoI6UFr2H9QiANGXavIwCHBurzKi8ODkVHFt82ka/HhqFLwSV0Roi2+Z7cwddOK+2xN0ygkwp15c5IvN1E08rdZaroBC8+5bL232/JJjAzclrBJu5/UweIDgKWqK5UjHX9ovkvZZPQeAG5UTt+ooP7/gZ4P0EW7+uiKKjqHpex/fTuFdK6UzkfAQ41t8RoF9V+spLaqaT6sIMfRSX+UC2M28olLWIwp0HIGKX05EIJy9SMDOUj+Iullek4GZ1rYmBZd1w5jFUSX17MjL30w2ZX21Z+p5WAYmIYXiRLXaM3hEAwhP9EoMcOZJSqr7b3s57GFFQ/9RnWvKG7sSTU/vvaivYPcBPOy0xxYV1UpDZf1pkUlW1cMqSGnML6Zbcvf46mupExpjVyDUdmEp57Uvw0X25IftRiB1wsxwGfQowqv5y4D1n8qUcPWyeca06DBx0CMJ0tz5jG4ucNwnAf50YLBRO/WK7Wvm26D9cozqpw8vBjIzQMMWyYqbxVVAiduo5fQ18Qby06WFOnpDCuVnjQohrT8slPENJbWVOMLAzPsR+5j9l9Iv7HjTlnTP8pbxdIHn5CG+wZOPUgWRXqOlPbhYBePLosZeSfsic0VPbyD2JRPCWagzRLbB5HwsuOEXLbc9yb6HqQxs0LeucO+U7xJta2BYluZmSaQMBmsx6kw0AHnKFHMTSytVA5spQo6HLeobSCGuiYv1kkgH3xnf9wwD3jNzsK7MqXhSTEg0Nq7Mq1JtVY5Wke4W95ruSPDBoc+X5TBvIvdZUHWBG1EUwx826wP8hWENdB2IY4J8OIQfidORzQpm3jmt1O1WzBgC/PFgNY2bZf37C9FtWFZ2vF754/Hbp24R6K6akKvnN8Lo0GvLYdI1K7ak5bi8NYWQ78iZnaO4zc46Yi8Iy+p6wzzZRyCR7zslIN/ZV/85L4lMj2tL63qD4/qx9apSLCBAqSM7I55hWqr/bqukOiyJDMMqTPNSH/Ez9RYCRrrKUBmz8uRfgGM9ukT4lOTBGu4zwmiloezOIy9X6m5iL+sZyuIMaAXSauiP2SbZgMwgZDF/i37Lis42sqhvxZCzB6p3LCxKNvIPJvg+GRDQDOXVjWsVph4NRrYPPoZqmTZuKpbYN/Gc9rTigosteNQlVxqemkHxveoe68TxT9c1b1CKGhixE1rfRu+di1tr8WOCTiY7iJaBU1OMTcozA+6NY152c3j/UpzITIC8Kly2R55RSW0W7RAOphPvUMstmsr3qCaN+zvrwYGKP39l+nyOw9af+nsvl+Z39COWQLG9xO7YiwnaK1pY042EUhRg5Dg20CZ5qc/t+nmEGX2Gxr46c9iJ+rTJXAjIecQyvOGRHmar5rjuOCpnCcDWYv80kHAMdx5yJfN4+FxjMVX/Cjb6NVkrM21iMtc57c6p+K0Nqf0vuqpfFt9Qg2hUebXZr4QXTmBx7I4DOFjTMYbs/mlGx2Xas01tVGqZPdds1JX50pKzfSXv22rpQUV2gbZmhg4uhUqXJ134HoPrRXjjY87iOoXR0o+elreaI/jBjuQnd4PswO9444soqgxR+jAigoQ/+hDn9sqitP9eWbZV4VaiqFj3rXM+BjkJCjT3oseFjJAvDLyI9ptu4YQkToOc5GtWIpAtgxQqFGI5czY9uHs0LJWgbMyls7SBi4TG09V875V54nLek7fbgek3XsOVxDBiWxNgGoNezJ0/SUtCPUzO0WrwST2tBDojah+luZWIvF1M+Yh8TRkbBY3esYv+JKDAb/YnjXFqEtVYtLEBE/Bac0Tc2TOu9EjrUw5f8onfda44VOvwk/8NrOGkqdiTsLWLvYkfJZVtL5oKuwzLn0ranF6hQnkSSPMyu90WzbqswFu6FuYGmIgAWi5EvRF7mfWkMopldjeNBc2UJ5wlHTRgo4UmX00NYmeUWkQ3GpQgJWbQZ7qD/Y76Rt87neMAFjJWzZ3D3Uiho2VD72ecaIM6NmNApgZaF4QP2ztnv72br6LXCuqpQPzCeoNjNvWcR/RedaOdMc7NroiETSVmlf2zKZeWy87tSxBrZJLaLDK6fqpEPNh6zrv0AxtZj0LVvAUi7vVzn/Q/3PHrkYuxrjgM7rz7LdtN3uAlgBIwh+c4p7KucaqvfSXjvHXTMPTdXzkiJbVprMAf9zQ8Zvcs022gNZX0uhKNiOhqSE7GNkwosoZRV2/pM1ixNmU4+sCt1OIJwGGu4efLUfbPElUjeAMT0ZMd2t4ZoaTNIgpX+me6Ni0YytaO9jX5srMv2z943u5j1FMrdkkqfjyxRD/J2lgJ4KaMgA9bbVK2akEaJsX42Unz4CFKayX7rfKngZlzmn8CmVIDrXovCkf4qnAdoxz2a+QSqNkPHFaMGs+Tab70ODJIQRUZdotHT2Ghh+V/fP0rKFKtEMSTIYtRk8iZcTmvrFyakGCtfJBxTwWfvzzk3tQgWkdeZlsKYAxdkmmrYmQEJuF9D2i2GPGNmjDzYAZplxshcBinkUNwWjbiIF0labKIc5l3QSkn2yc9zSevB2vwiVi2WzJz4nXne/c/J6p1gC/jLcAX+df8vpQSWQ96ucGI+r6nWjQw5znuslkfW4MGSaVGBnOgR5Wof9maHSCyV2IJv9jPufbezyk2gK1PklCKYPjzio6C/znEA0DtoF1YdZ9nt+uwz+BvNTqsP8gYNd7BFG/sIN7whR1CBj2YE73/cVi0Ltv2rO9yidcurnbBE7mvM5TCcgKV58ttknw0PLrZRnQl08/e+zASoZv/mvhuBIvOwgjDhebk9AArNQlktotW2kaPBq4bujXDDvYvw9n+NKffEQdAXBMVxDuppE07y7GeNnW2Uu0K5lEEIzm9HshK8uwT1lrY4ZYjseMlJhT2HE8/iM6DWHdANcfT09ajaxhJQNzHbBPyGHWaYcB9zBKwMSYBlOKldMTbn8OsYOkx+SxthP/7pEkYbn7HBw7TN7e5RpI7x5RBSde5eHi7tNonRzHofeMaS6nnSyb+zfFXvt3t2lyN8XxObo7l9bskrrLoLfT7d0v8PHb7RFTvu05jVbhnHwgnI/BLhbeZ8hA8MvBCEa+l7fohrgv5p2NuG/yy8R8SWxJQtihaXeS27WL35rRrm2nw36ZlJuXZAUaqJWTDtVFSjWSKi3nTuUApqD8sBzDZM4KRRc1FyBNvUoH8onf4ASdv8hYpUG+c+SYsFuH/+Qchnkiz8+x0qq4K5d6m35wb6nT29cSXG6INO8xbd3CQfr2nVzfuhUZ3KasY4bk2qOR82XZOSPrgHzFH0OOGpqSTxz389R7voqIeSxhrWael8/WtDmgYARjrsYNGKM5muioHOeMF3B42ENaEL5fOiyYMwFPKSchvTdEzSOJ+qCKxAFBw1NNYkJJ8hde/WNWHA3ejLPcKnOvbVJpvey4uRl5YICy3olykPxgTx2pCeu97iZ5XAhvxjOnPKE8inIRHFu1fZtfeQYd1WMFGlvI/TWk82Fg5k/P0pxEsV3jpG7U5477YkUj2kxuGipMRcAHtDzsV60ExUF+GBQxqu3Oo77GWtH94Ize/saYamhNVPD5E3ZTb1bjlz8UQvUrbICyQOZCXliI75Ggoa16AbWyiU9j+XWeydK6wlpasf432nwdz+K4rxSXr+GxeEEGAZP7PqAMFBw+aHufMPwtuI9vh/LtHr7d32+1mtO3NbSpkUj2AWQ3LAATkZTX0cLGFmOl88CRLvPlKiaCNHMHMBeHIOHPZtGbjrnlZ2nrJrdCQQq3D9bMsdesmI7SjRlZt7P/AAPI+2msvjgzyv+Yurd9V4O3TKnyZ3sfqzJWSvaqwcCub+sZD0nfianrwWH/2iHVvVBzsrEHUnaGQ/1ZWxiSAHdQAa5N2CoxNrBQJpJg917Z9eTEjDKxQ0ys8kAHKa+ZYZrS2kDfGrx+hYoUC+oJluV0fOLdrdrmBGw7nsWJAJ73X0cTFeXBdxRL4TF0t8nQxwaDXII9o4h90rLEgZ6aUS+82UFwGz3taRNvqUdbnOAnGrKFjLVrbOicFrmzM6WOWo23fIq2DFpvMHBbB7thzqtqLCrfEeq8xOT9JWt4CNXtA6hpPgGHOOw2xqRLXXfJX7e26+valQXKS9ynWGEkNl3Gdq3euE5La/QwX6N1WH6Z1XnLlVlflYsTIxA0rr9S1fO2rI8S2SIzu3yy6JpV8WFE/IsQcDPxgs6kdVEvyTcrCEZwK8s9dM+N/YBIOW+LZlncF0pHS2r2Av6Bterjj72DPEIac5yMmYY5otb+NARHkEEAfGChXWRoHkhqZtOnRqvAZl5VgNin+WuacFMcpxyxXLuqHYjZs6SkPTeb3tsWj/dO01wvOvwNHqRTsds/2BdbSY9bS1TvXVJKq+Gl01XPA+JL3D1Iqr3OYa4kRou0UT18+NCJ7gcZrCzZ9RQwnObP4hkY68d4heus/AwZZI5PMvPQSX64z5lcIo9HGr+LgkBbbpnLAzqfPXv+7Jsnj18WbwhhFJ3tARal40fK0py+9tiS0vYdAaN3j8uQcx4ZGMrczkKTTxlG3qs/3oeadPM2EC3LfqFm4Q+wG8j1vPkBvnF+C/sP+zV62lPJbB6cAogwV85eN+37DfP+89WSlVssgvTtKkP69IzZv8ArxsnqWPXrCeP9lh2bPhvlwSdWUzeeTarO8L0u11BwTjtEN68nZukxQwejwAQ9zkFhc1DFD8jgbAPpbESzVKEj1lRhOI7WS1Xy27dWeKoe43j8DoH3aWBUmDjEyc0IYnrokqhnJjf9Xe+HnZJ7lY1HLqtZ98UbAUqeVBgHvmxiFM2nNJje9Y4cEjwBdbEzgsune9L7nnjVEoK8ooADmpzjAV96JBVNAR/KaNjEoHixlgkinTON5HqdntChDpmuoUTbxFN8oYA97w727wt6BvdxhwZwFNPFeLDXO3DQSwrCy+gWA3nkZVUjStsT7OqaWJ3q4zOED2FDpTY9RipJfLvLb8Iu9zskUFlLGa1P8o0P5WnOP1I58ztkQEpiCj+ksUDaNE2JH0iGZBkDHZ3Rf8lUwNYBDU0csMeRq3BPhbL/IWuTGHXILUELijBQB3U0nuc4uVfBCVhkr4KU0NKHzPpOTh+HNAo9lNgU1EJiiWCPCAGt9R4Ug8UjEtC+1hISwud0u6R8MNc2W/UmGOJoWFPlSbdn1aUOPqroqKCBflsFAiDeQW4p8lTbsWOS3j90G53ZMC4ScVYdSyAmPTMU3rBBAQ6si4/kFNuqSRewL83L9qVh76D3gGTee/jJfX7uHx58Al/1h/d7NlwfJ8GB14Sug1sia6/3sISp42gFak/GwFgQneKHZMFrW2d/NblUMBjIQ9oUWhaTz8u86ZH3MmngVugs1Wzpsq5hJi8WeVu3574lXtrZzQyaQvvz0vWKg1fCkmazMiaLkTjsDhHtCPqZk6oJCBI4LEdOKEaajw7oRTUzNldTlFIk6FeCD7lz4O9wL+3Dw5OtFJG5kfXowPnYeKihqiMIaP6HwGVKe8Lp5E8ESFKzQpgSnPZElV/2ehrmhTGIZGMQqg9ztOv7eWfPh2k/EgdfwtuXpyMl5PQHOt10chIGXC2Zxdu+ckdfMZdu6NxPF4z92JDvTd8Ew+c4oiQSKzfsy0lHqEiQjNK9Xp7J6yyGhMNrmzrMM56b/fyzCdz24T925jj40NK3MrJHGuwDZilKkZNSJKWiCIFchjrcWUWmVcRyRPIwDaAWAUAH+58cftLqIBEurXYYAKDFdQidbD15i4C64JeQSJAFaLZVccHoQPROl8hk3DR0azM/2eusOsl2FcBlYM5AcAmz2zuVBJOMS5Cf0BcjFmGMQIcoXEp6kNN+V5+crM/GzFgXM23ITbYMfXkgCGyrVNnXsnEK4MbMGz5I1hqgYZ8gbCTVwujm1vfWc8LD+7uYupyjxfwj2xxvChljxBAtTc5eSeopfqwgopnNJUIRh29LPtNUwpyHvh0wfa9cg+gMh3eOgwd4fNfRWNP3OsuXq2S+nZGNzIwPrR3pmBF2zZdHMF+dxYOkrUtnFQOb9CCZXEVjnQKE/AXL1LUS/5KnUWRRCIhR9Htx2dLfFrkwUrJmNz+rkM/jYQv8ofegS4bw0GFTBhHgsGfuHsFiswmoh5LMRIn6rMVbQf8QIiyRGpy3KPBkg5m5tsS/RY5zSybT9HSwGXQ/mFLy5H421nQ82/JSsF98eYWNIzZLCHnLovNbLYefO053wngZkSyLXyAuLGGueM38HhTJw9TVC3uLtS9rlcDcBgSlAV1dYpokqRFZVPzXHqd6a5VXa5HOYQytQnoKqvHlgkATcuvkoHeuduzsmOAz7djrF7nb8p1DI+Tljj+oRFvGYW0XGYwlA+h2sfUe6TOBk+xlmyJdq+Z4xQN1h/vcmq6kK5OAhlEneo1W/X5DXvTnxXdpw2Um0kzuSlvj1j/vzVfvmDdgIbngQiRxy0z4seSLXe3i6kFxlZZAAZ4TJP2MZmAzA5W8uzDKYba3cofiJM/jll+fYqPZ9JVbRLQZFz5mbDVTlNggttkazhmRuOPL1DIEvWObNlHNKWTNNzCbJmlgqkOIoffAeP3GSRqvYdL1qDR3wmSyWtWYB+3NEepUiBiUWJgRWisOvHlEl7IzFgoTbVU8cLVFRl1R0wf9rWaSQHC0SleU57QJAmgk5ZKYmCdYFA1SJTCrhvq5/7UhqAtxFL70ggvreYErDyPxXDJDuqTju5Ik5gPe8ib3sGY+0lBEkLqbXHyjNgfboqQLyOcGyQfifvdgdtlZbu45L02X4soIRrjcRMfC4HDT3gJi9aLMDt8q8yPLQ1sYcr9zML+UwkzLcmQUT2LfHiwK7TMXyKBg0tUptAHkVQctAnqHkO7NNTf86Ty+Re9zP1xtFwv43g2SwcXRy8v+dDHgvQfR2jHqFo2fzuWllDdeDC6+u+xffHXZn3PacI1IfEzj9jjk7TxQGXGHA9+fJlROtqcIMbNvtOdw5L3AbaCXbRFRk0GvbXbGt9vTe/vdTcNO4e+F2jk8SyjVEgAA3A7Nm0b2HS/Ct1j/qR1FU0PLcCOJGfpWIswLQhTIAWdtvJL9npQ1tTLty5YIh8zzfVyexHZoa40kaQd44FveICHF00QmpXd4TTuyc4bhVCpP9CfY5hv5HZLAmKHgSEzE3GCZzYItk6Qdyinv3pg0oa7pm8otFYbpWA09f5boSzeq6mJlGbT5coTnYV7DkAwbklKjiJNN57p15stj3mXrO0ccOd3IEx5uMDAm9J7hztyBv83mew/0tDMvh0qqV6iOper1KEcZpOBghMouc4RsikTRu5Im+vvvd5Eqmqx79/uVgaVB/LnVqlexBZkFQ53uNJwXiT9kmenYmrMr7U319Y02tlyfzhcq7/p3SEF684WVI0b1OdW7fjvJEhwwKlANoxZ0vGMBNnWqN8ywmd2qhUR7tMpc3Ds8Vhdj65u7tJoCm1gAij+oIHlehowtT2fxKRLWBLSqHhx2+Q8Pl/WHh+ZRWn8kTZakcP/ipN/dn4GVLM1nKp/tC5ijyMRttttILo4i5MaruKTFKUivOFjfgvboAdMrUeJx4Z9s4yNGjRm7yy7O1o+fSSurD6wrPPMC1LN654lzNPeU+wMWqLAhRrUaheguE+DzzeX3qwtnhkMk3dnDddVu2aTFSrverk0wtnu2O3JNwLs0ATNfz/crLqk8ESYuxPzbV+PfPvviybdU8fiyxbZFZjrVz3v6eRZNllrwRgvC01hS5Ki35zPnms9cgBYzETPYGIFttuveBlrcNGoviqWkzbsMNFzE5K2nhILYOu6RQT0g4zrvUtofXa0a/dNEXpNvKkOpfMSaxWS7ZRq9UW7DXHlqLJzD/ArUUbmaXKmidcwpmfejyoE48EX+053MadK6uvHOnJTkH+QzY3OHij295o4JrmiGFS+P7bfZedsOGukkI3FpLi6kUkWWOtNbBErjuZela4BJwqlrg8xFVOqh9nLUtuTuXFgV4ejvoNmp0CqrKQs8DezQ6j6ZZljXe6w5ntfCLlyUtmALXPHAI67UDRnCUiNxfwfYrdVSTO5ebqhsicW11232niRoJRNp1dja0lSCNTur7srWkvoc2U///bdHZQDfGRuZJtmgTp8T0fjvI7Uden+DA95nYMNM4af3hst91+p6VxZJz8hwPXVo4Pgw5nccevGr7JyA+X7bFSdbM+7cxjs9kwintvMot/FOT+ph9q7LKCtb1PEaese2/Plv8D9Y8/p8EGjvMPNL7M+ZgHKHjZdn+LFfpzAuj2bqKR6XlCsvhIxq030LgSP226UVLQLiSvUzjX7ZyG5XyNo/U2q3snpTP0YKQoiAsh6RhuPz8XwpMhGii/ZVRJaszKKwGbgYefbiSau0xYoForL6P/YwjlYlAtkyI5wx31QCqbaSmNTOgdUTYWSzgDRJSqzvrmKJ3s4ZwpxXuk2KxhyOUvZnZ2fImSRar9FQ/riRjtxQmnLchqHoIm6jCK+ovcgkmuh7Tqd/Mqvi9RIIOo2cLK1E43B9EeRMHc1BKyELs5yGobQ2mq/tLebmDbZtelS4K6YUGevwQhOhpffxrhg8Y6+zFPHJSchUaN/MFd4D3ZrQwbG5F8CoE+vaOeNBhv5MmElgrCUl/++K2FfyBlcHDwZMNoXduVdb2BwZ2pqGzo7a5qdG2ZZTrZSi4XaMB8TxeqypAwb64zhRd4gcj+LhbjjHmFUk9LMS3VKu4SbaOy5vg5WKVQp5pC8sjjtkIsHiOI/ccuizDHTBqA+63ZID0YHY3Rc527/L77I7tFrXb3HimPfJfleWtd1N7jrNMAPEoUo+0DqUobLYc6Cyjq4GKDT/YTBJu7PIigQyfRrcHtrpdpIV7XkHncK0IemPJT8Loe6XYTVtMjzU/Cy1FsplbrqLN8RJV8KQncukVypZavxw5f5tVzTAekIyQbQ9Yzo4SAYNvcACgLyQemNXEUuoTYsrJw9JWArqnGl+FCcrA4PxMC37bTmNHBdJdruSr+g4TwhYhDe0rj221cMgygmv3cOFdOgeSiDTmxx1mgL0ZadcXTPdXXM1IKTjTLFt5f1W0+qEx1FxIGiN7KzSFrOVkwy1ojqPbXshGhLnktnh8NMW5kWt9mIXU3QiV4khnch0fco7FP/sySSC+0iqd6WEFESfUoy7xj67U5Vy+QLl2/REnx1hkzn6ymzp4lk1jeJFkHL1pBK4ji/7xs3NeX5Sfi5tnXJHb5BpjiiiHFGqoe5kMImnW/zvF1wv0kGJzUSSMEhw1Ngpxow9efWYOXkfP31yVBAXusSO7repRnIqDOtNmCh568R8RPHhSOQHpVkYLcXpQ5N583iEM8mxSdmgd5hObQKHYn9FykhoMcacYSyjwFzZyJRYBYw4pgYvo5beSpaY++jqFKETL+cgr2iHKcOytfIBvC0XA/BErQyQQUlF9JHBJ4cyVHhH1+/ubi2zwX+YYr0UneQA/GnV+aRQUBca+Lz2XUqNqvZ28ri5I2gQzqVFNhhbQ6SVsUEQ7Q8Sd2+VilrdRW4WixVmyXtJFVkxwj1qWdSMW6KVbqty9h4rHQRl6iHvEfnZ7YBsTJp2Cz8lKkHQ96r5WUvdsvWKkO428ms2XAkx0DcxKndeGoR5U7Ekyje55ZfleK1xGyHehbMmx+eEb6ZgUB+I0wZW9igXmBpEb2WLYHsobxS5q0K8065Oa7bcjE/LZkihRlGw7ZiNEYmgRBNVo/APU2+/h9Jv48ZgVuZ3l+c4TEo71rLBU5093WLJtG82rtyir0bry4Mq7LmtSTGpB9N41bwAV5Oystv4bBF0iLqvS+4zK8fNeOIsNHFvWSlDclffmj62IuZXAi5l7bx2siYZCZCMWeU7vSJLM1pA97vOjGtsnSxf1wl6Y04SzCVXPpcRUJNXAxEba5MpY13LKGd0Uy5ryVEkLmSCn2r0VyLOfm6wvSPuGWFO3e+ZmxtDn7TMzfPl3heLvJlJcRFDWejixQsdMUHbtjpqrtab7OO5gaeUi7508izd8lBqvN8UxDOxGyLBkx00Y1od6faakJ6OOUPXeFgOWn5zicSo3aCr1HiC5hwPVlegrTLFQY86QknQ0GqIH+cArt3cS52hZscod7PbhIUbfa/qd5TOyrrdSiLJutD8TglUUl2OxPPSqjH37xs9pqpVuwd1sAvFpNFIVlWRqo50ooiOWHVfNJGqiGRE0dF3qpp8pGrLvO5XEsB1X3STTu2vVFl5Axb/1TSVt7dUFaHgzjGq28tXjz1ONVOyhlDaWCmNEStitq4PNX1P0JXSSSZXC8GnXLLQ5gFN9Y2/0ClU7/x52ZGHUN5MYcNyVmd+imZRmSFIbiddCdesgibgCyA03pGTv3+QWzRsfzoieP/fEH95iyU97V2zji06VJa/eTFfOwx5uW2iBvMCo+owMYTBV+WH1rIhT2/esXV7ke1P0eFiwwgwHdmZjcHSxEyVDS+uU7KbHKrQ9GlSSLiGBirE3pdbELrVVKzio+p7uqt5F6vLuxerySXAvNCLSVYcE0t+xQtY5tgOTioOFrmFvUZ/7B0hEfVujQWessHKpPKoDH26wDVu1JDm5Wchb1B4dO0x6317lrPtFJl5S//gXjiL1wrNaea4qemRWmLJxjaW7K9sVKNjYpusE01PYPJ9XWTObj92dvviJkBJS+zcSZA/0bYqB4+aUd3IP3W1/vpm6uf48phVG0WAKAmHmqaafG54/D7tkWTEqM19PbvGqNq0va91UUnwhqx/e5Iz04ZZD20v9grLYd/oJSUL70IYy11kohoN+4ejurPCdPF+TDZ2ljbkFP4gzDK0pYxWoSwe3h20KzMAVylCX9p5JknsACaxBoJhyscfFGBZnPo2yMaJ6HFieVBgSN5matd51OSXo9wzk/wvnIAh19NQv3lvNveZ5jfXPfRdkWUub7y44moIv7NRCO/l5eDi3XA33B2JtqLPHxPzow1rXLaBW9uLV0d7R68ev3yFL22/mbPCR24ezqJBBkNefzagUNS2JZhgBCoMLkyOyJnu8IgVlC6l8sEwhEfIyJEfFA0OxG4Fnsv9R/YCJg5N/Koeyd1MplgbsPUZpjs3HlOD1eCCRKv4u+yfDoAN+ANeA68SAWHev+/AvH+/hAsgAKHlA9ATFHQMmryQqVdMLtbbGQMPB88YENh/9WKwt99//vLxoIdey76FlTZv9DTEDPFOm8H+8HQ0PH0zelR+edjjJN/cwul6AXJihOf9YWrRKy4qynkq2igxBs/Pc5ZsbFyOXty8alRjGp6yWMsS9Pd6uUXZvOhk+l3W/YSPuc/gsP5S8tFC7SqJF/RNX4PsxY+4U/IkNnP9aLhf0Phsaeh7xnsEnJlppPEPcvesrYi5b5B68bKj3yzMl+3KIjm8jub0rA8lW6cgOXwfLrvlSTYTjWLOP8NmAKS5JxIiyCA7wQdIUldkX03th31gFWFhSA/YTxBTARwMDu53h8vGtkFa6QjPwmV8Drg3HPlBX8NocW7A2rGhJYM5nM+zNoqXkJmSgXyi4wcPhyHGl1NXSdxUVfcjgq9+wfHqzSPt1O1x/y/rMVdahWWuLVIGvMBC/2e4YX5siF00VpJGUcLpF4w68eaOk68B5jpvTOMvXVBloxPujT67MsQmR82/0L80thoq90fYrrmOFp7B14ztw31LC7nMAGC65hbavz7vKIxWDIU5RfR4LcPot0fi7n9NftEm11OsGq5buXqJf0zPx/nGnQtgO+WcWHP/C715zmRd63sX8sKuzC531t78Ejam4MLq1oa7og+TWwdtzuvaKrworquVxOvHZL7nw10R73ZLkh+aqZtCW34hSGgD9audNNWOm8XuDY8N51a6FDWVc8lrPb35uZXfnCxkiBniZg1XI+P/aCAa+TqEJsOnvTSxfOGeIwx43gUGD3UBEHrY2Z9fpijJ/UTleo7LC/Rs78+0F5rDSdQIB3qzIWab99RJhtrxWFz2xriTBdFIY3OkMBe/4vYgiBnmps2QSkxb0nmcHG8Z7vyCv6xCLUQA+Gw2Ds0z6x3JK1eqT5Tu/Gve29sTjDJh8mIz8P/dEa7slKvCMGdYe2sESSKNBuOZcLujvfe7dUNrZvdsgmSPLIHqdpOCY+Bf345kwSxXbWyR2TGbWiQGEdRPnLF+avD2kpk2jSdN4Pp6DELRo1hPF83CiUKTjbOUixPF/NJyb0bLY1YDsUSEch+ExnHrd5N7zdzKjiQUYccxNsuVRYObbmUz8U/2LUl7X5zua/zEAie+ROal3JsICX86VS8iQKnfiqN6yc/cGu1FIqR+2jbRQlCunlQSGwGhMPnf/o1f9NTa+X9lct0nwKkAAA==")))
open("/content/montage/logo_locapay.png", "wb").write(base64.b64decode("..."))
sys.path.insert(0, "/content/montage")
import importlib, montage_auto as M; importlib.reload(M)
import torch
print("✅ Installation terminée —", "GPU actif 🚀" if torch.cuda.is_available() else "⚠️ pas de GPU (ça marchera, mais plus lentement)")

In [ ]:
#@title 2️⃣ Réglages
#@markdown **Couleurs des écrans de couleur et de l'écran de fin**
couleur_1 = "#2E2C51" #@param {type:"string"}
couleur_2 = "#0AC2A4" #@param {type:"string"}
ecrans_de_couleur = 2 #@param {type:"slider", min:0, max:4, step:1}
#@markdown **Sous-titres et découpe**
position_sous_titres = 0.66 #@param {type:"slider", min:0.40, max:0.85, step:0.01}
couper_les_hesitations = True #@param {type:"boolean"}
pause_max_conservee = 0.45 #@param {type:"slider", min:0.20, max:1.00, step:0.05}
#@markdown **Son**
effets_sonores = "léger" #@param ["léger", "aucun"]
ajouter_une_musique = False #@param {type:"boolean"}
volume_musique = 0.18 #@param {type:"slider", min:0.05, max:0.50, step:0.01}
#@markdown **Logo** (cochez « autre_logo » pour charger un PNG à la place du logo Locapay)
logo = True #@param {type:"boolean"}
autre_logo = False #@param {type:"boolean"}
#@markdown **Transcription** — mots rares à bien reconnaître, séparés par des virgules
vocabulaire = "Locapay, démarcheurs, loyer, propriétaire, locataire, Cotonou, Bénin" #@param {type:"string"}
modele = "turbo" #@param ["turbo", "large-v3", "medium", "small"]
#@markdown **Vidéo déjà sur Google Drive ?** Indiquez son chemin (ex. `MyDrive/Videos/rush.mp4`), sinon laissez vide.
video_sur_drive = "" #@param {type:"string"}

R = M.Reglages(couleur_1=couleur_1, couleur_2=couleur_2, nb_ecrans_couleur=ecrans_de_couleur,
               position_sous_titres=position_sous_titres, supprimer_hesitations=couper_les_hesitations,
               pause_max=pause_max_conservee, effets_sonores=effets_sonores, volume_musique=volume_musique,
               logo_en_coin=logo, fichier_logo="/content/montage/logo_locapay.png" if logo else "",
               police_serif="/content/montage/polices/serif.ttf", police_grasse="/content/montage/polices/gras.ttf")
print("✅ Réglages enregistrés")

In [ ]:
#@title 3️⃣ Charger la vidéo (et la musique)
from google.colab import files
import shutil
os.makedirs("/content/montage/entree", exist_ok=True)
EXT_V = (".mp4", ".mov", ".m4v", ".avi", ".mkv", ".webm", ".3gp")
if video_sur_drive.strip():
    from google.colab import drive
    drive.mount("/content/drive")
    VIDEO = os.path.join("/content/drive", video_sur_drive.strip().lstrip("/"))
    assert os.path.exists(VIDEO), f"Fichier introuvable : {VIDEO}"
else:
    print("👉 Sélectionnez la vidéo brute :")
    up = files.upload()
    vids = [n for n in up if n.lower().endswith(EXT_V)]
    assert vids, "Aucune vidéo reconnue (formats : mp4, mov, m4v, avi, mkv, webm)."
    VIDEO = f"/content/montage/entree/{vids[0]}"
    shutil.move(vids[0], VIDEO)
MUSIQUE = None
if ajouter_une_musique:
    print("👉 Sélectionnez la musique (mp3, wav, m4a…) :")
    up = files.upload()
    if up:
        n = list(up)[0]; MUSIQUE = f"/content/montage/entree/{n}"; shutil.move(n, MUSIQUE)
if logo and autre_logo:
    print("👉 Sélectionnez votre logo (PNG, fond transparent de préférence) :")
    up = files.upload()
    if up:
        n = list(up)[0]; R.fichier_logo = f"/content/montage/entree/{n}"; shutil.move(n, R.fichier_logo)
DUREE = M.duree_video(VIDEO)
print(f"✅ Vidéo : {os.path.basename(VIDEO)} ({DUREE:.1f} s)" + (f" — musique : {os.path.basename(MUSIQUE)}" if MUSIQUE else ""))

In [ ]:
#@title 4️⃣ Transcription automatique
mots_bruts = M.transcrire(VIDEO, modele=modele, langue="fr", vocabulaire=vocabulaire)
print("\n📝 Transcription :\n")
print(M.texte_transcription(mots_bruts))

In [ ]:
#@title 5️⃣ Corrections (facultatif) et plan de montage
#@markdown Format : `mot_mal_reconnu=bon_mot; autre=correction` — ex. `Locapé=Locapay; démarcheur=démarcheurs`
corrections = "Locapé=Locapay; Lokapay=Locapay; Locapaï=Locapay" #@param {type:"string"}
mots = M.appliquer_corrections(mots_bruts, corrections)
PLAN = M.construire_plan(mots, DUREE, R)
print(M.resume_plan(PLAN))
print("\nLes mots en MAJUSCULES seront en gras. Relancez cette étape après une correction.")

In [ ]:
#@title 6️⃣ Montage et téléchargement
from IPython.display import Video, display
os.makedirs("/content/montage/sortie", exist_ok=True)
base = os.path.splitext(os.path.basename(VIDEO))[0]
SORTIE = f"/content/montage/sortie/{base}_monte.mp4"
print("🎬 Montage en cours…")
M.monter(VIDEO, PLAN, R, SORTIE, musique=MUSIQUE)
M.ecrire_srt(PLAN, SORTIE.replace(".mp4", ".srt"))
open(SORTIE.replace(".mp4", "_transcription.txt"), "w", encoding="utf-8").write(M.texte_transcription(mots))
print("✅ Terminé !")
display(Video(SORTIE, embed=True, width=320))
files.download(SORTIE)

---
**Fichiers produits** (dossier `/content/montage/sortie`, panneau 📁 à gauche) : la vidéo montée, les sous-titres `.srt` et la transcription `.txt`.

**Astuces**
- Filmez en vertical, la personne au centre, avec un son propre : la transcription n'en sera que meilleure.
- Trop de coupes ? Montez « pause_max_conservee » (étape 2). Pas assez ? Baissez-la.
- Sous-titres qui cachent les mains ou le visage ? Ajustez « position_sous_titres ».